In [1]:
import numpy as np
import tensorflow as tf
print(tf.__version__) 
from matplotlib import pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import json
from pathlib import Path

2.21.0


In [2]:
DATA_DIR = "data_uploaded/"

for L in [10, 20, 30, 40, 60]:

    data = np.load(f"{DATA_DIR}/L{L}_ising.npz")
    """split data into input and output"""
    T = data["temperatures"]
    T_c = 2 / np.log(1 + np.sqrt(2))        
    labels = np.transpose(np.array([(T > T_c).astype(int), (T < T_c).astype(int)]))    #create labels from temperature
    configs = data["spins"]

    rng = np.random.default_rng(seed=42)
    idx = np.arange(len(T))
    rng.shuffle(idx)

    # Apply the same permutation to all arrays
    T = T[idx]
    configs = configs[idx]
    labels  = labels[idx, :]
    print(labels.shape)

    """split into training, validation and test data"""
    train_conf, val_conf = np.split(configs, [80000])
    train_label, val_label = np.split(labels, [80000], axis=0)
    train_T, val_T = np.split(T, [80000])
    print(train_conf.shape)

(110250, 2)
(80000, 100)
(110250, 2)
(80000, 400)
(110250, 2)
(80000, 900)
(110250, 2)
(80000, 1600)
(110250, 2)
(80000, 3600)


In [3]:
DATA_DIR = "data_uploaded/"

BATCH_SIZE = 32

def build_model(config_size, hidden_nodes, l2):
    initializer = tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0)
    glorot_uniform = tf.keras.initializers.GlorotUniform(seed=42)
    x = tf.keras.Input((config_size,))
    y = tf.keras.layers.Dense(hidden_nodes, activation='sigmoid', kernel_initializer=initializer, kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
    #k = tf.keras.layers.Dropout(0.3)(y)
    z = tf.keras.layers.Dense(2, activation='sigmoid')(y)
    #y = tf.keras.layers.Dense(hidden_nodes, activation='relu',
    #                      kernel_initializer='glorot_uniform',
    #                      kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
    #z = tf.keras.layers.Dense(2, activation='softmax')(y)
    model = tf.keras.Model(inputs=x, outputs=z)
    #model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.summary()
    return model

for L in [10, 20, 30, 40, 60]:

    data = np.load(f"{DATA_DIR}/L{L}_ising.npz")
    """split data into input and output"""
    T = data["temperatures"]
    T_c = 2 / np.log(1 + np.sqrt(2))        
    labels = np.transpose(np.array([(T > T_c).astype(int), (T < T_c).astype(int)]))    #create labels from temperature
    configs = data["spins"]

    rng = np.random.default_rng(seed=42)
    idx = np.arange(len(T))
    rng.shuffle(idx)

    # Apply the same permutation to all arrays
    T = T[idx]
    configs = configs[idx]
    labels  = labels[idx, :]
    print(labels.shape)

    """split into training, validation and test data"""
    train_conf, val_conf = np.split(configs, [90000])
    train_label, val_label = np.split(labels, [90000], axis=0)
    train_T, val_T = np.split(T, [90000])
    #print(train_conf.shape)

    l2 = 0.05   # regularization strength λ

    model3_2 = build_model(configs.shape[1], 3, l2)

    w_init, b_init = model3_2.layers[1].get_weights()

    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=8, min_lr=1e-6)
    early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

    history = model3_2.fit(
        train_conf,
        train_label,
        validation_data = (val_conf, val_label),
        batch_size = 128,
        epochs = 75,
        callbacks=[reduce_lr, early_stop]
    )
    #callbacks=[reduce_lr, early_stop]


    file_path = f'models_3/training_history_3/L{L}.json'
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)

    with open(file_path, 'w') as f:
        json.dump(history.history, f)

    print(f"Successfully saved history to {file_path}")

    # Save after training
    model3_2.save(f"models_3/ising_classifier_L{L}.h5")


(110250, 2)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           303 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │             8 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 311 (1.21 KB)

 Trainable params: 311 (1.21 KB)

...

 Total params: 10,811 (42.23 KB)

 Trainable params: 10,811 (42.23 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 338.1017 - val_loss: 187.8386 - learning_rate: 0.0010
Epoch 2/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 114.9916 - val_loss: 63.3392 - learning_rate: 0.0010
Epoch 3/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 38.1183 - val_loss: 20.4031 - learning_rate: 0.0010
Epoch 4/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 12.0219 - val_loss: 6.2462 - learning_rate: 0.0010
Epoch 5/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 3.5959 - val_loss: 1.7965 - learning_rate: 0.0010
Epoch 6/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.0334 - val_loss: 0.5553 - learning_rate: 0.0010
Epoch 7/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3701 - val_loss: 0.2536 - learning_rate: 0.0010
Epoch 8/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2043 - val_loss: 0.1780 - learning_rate: 0.0010
Epoch 9/75
704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1500 - val_loss: 0.1328 - learning_rate: 0.0010


Successfully saved history to models_3/training_history_3/L60.json
